# real conda, in your browser

This notebook runs **real Python 3.13** and **real conda** — the actual package manager, not a reimplementation — compiled to WebAssembly. There is no server behind this page; the kernel, packages, and conda itself all run locally in your browser tab.

[conda-express](https://github.com/jezdez/conda-express) (`cx`) makes this possible: a set of Rust-based conda plugins (cx-wasm) replace conda's native-code bottlenecks with WASM equivalents, so conda's full CLI works in the browser.

**Run each cell below to see it in action.**


## Proof: you're inside WebAssembly right now

In [ ]:
import sys, platform

print(f"Python:     {sys.version.split()[0]}")
print(f"Platform:   {sys.platform} / {platform.machine()}")

if hasattr(sys, '_emscripten_info'):
    v = sys._emscripten_info.emscripten_version
    print(f"Emscripten: {'.'.join(str(x) for x in v)}")
    print(f"Server:     none — this runs in your browser")
else:
    print("(not running in Emscripten)")

## Real packages, real computation

NumPy and Matplotlib are installed via conda from [emscripten-forge](https://emscripten-forge.org/) — the same packages, compiled to WASM instead of native code.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# FFT of a composite sine wave
t = np.linspace(0, 1, 1024, endpoint=False)
signal = np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 12 * t)
fft = np.fft.rfft(signal)
freqs = np.fft.rfftfreq(len(t))

# Lemniscate of Bernoulli (∞)
theta = np.linspace(0, 2 * np.pi, 500)
r = 2.0 * np.sqrt(np.abs(np.cos(2 * theta)))
x, y = r * np.cos(theta), r * np.sin(theta)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("conda-express — Python + WASM in the browser", fontsize=13)

ax1.plot(freqs[:len(freqs)//4] * 1024, np.abs(fft[:len(freqs)//4]),
         color='steelblue', linewidth=1.5)
ax1.axvline(5, color='tomato', ls='--', label='5 Hz')
ax1.axvline(12, color='orange', ls='--', label='12 Hz')
ax1.set(title="FFT spectrum", xlabel="Frequency (Hz)", ylabel="Amplitude")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(x, y, color='mediumpurple', linewidth=2)
ax2.set(title="Lemniscate of Bernoulli", aspect='equal')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"numpy {np.__version__}, matplotlib {matplotlib.__version__}")

## How it works: real conda, in the browser

This isn't a reimplementation — it's **real conda** (the Python package manager) running in WebAssembly. The `%cx` magic calls `conda.cli` directly, just like `conda install` on your laptop.

What makes it fast is **cx-wasm**: a set of conda plugins written in Rust and compiled to WASM that replace conda's native-code bottlenecks:

```
Browser tab
  └── JupyterLite (main thread)
       └── xeus-python kernel (WebWorker)
            └── Python 3.13 (WASM/Emscripten)
                 ├── cx_wasm_bridge (shard prefetch at startup)
                 └── conda (real conda, compiled to WASM)
                      └── cx-wasm plugins (Rust → WASM)
                           ├── solver: rattler/resolvo (replaces libsolv)
                           ├── repodata: CEP-16 sharded fetch (msgpack.zst)
                           └── extractor: streaming .conda/.tar.bz2 → MEMFS
```

When you run `%conda install lz4` (or `%cx install lz4`), conda's full CLI runs in Python — but the solver, decompressor, and archive extractor are all Rust compiled to WASM, called via conda's plugin API.

The solver is fast because all repodata shards are **pre-fetched in parallel** during kernel startup, before you type anything. By the time you run a `%conda install`, the solver has everything cached and runs as pure computation — no network requests needed.

In [ ]:
%load_ext conda_emscripten

### Install a package live

Both `%conda` and `%cx` call real conda — same commands as the native CLI:

| Terminal | Browser |
|---|---|
| `conda install lz4` | `%conda install lz4` |
| `conda list` | `%conda list` |
| `conda remove lz4` | `%conda remove lz4` |

(`%cx` also works as a shorthand.)


Install `lz4` — watch the solver run in your browser:

In [ ]:
%conda install lz4

List installed packages, then clean up:

In [ ]:
%conda list

In [ ]:
%conda remove lz4

## Try it yourself

**Native CLI** — install `cx` for fast conda environment bootstrapping:

```bash
pip install conda-express
cx install numpy pandas
```

**Browser** — open any notebook with conda-express support on [notebook.link](https://notebook.link/).

**Source** — [github.com/jezdez/conda-express](https://github.com/jezdez/conda-express)
